In [ ]:
import pandas as pd
print("pandas is ready!")

pandas is ready!


In [ ]:
url = "https://raw.githubusercontent.com/dD2405/Twitter_Sentiment_Analysis/master/train.csv"

df = pd.read_csv(url)

print("Data loaded!")
print(df.shape)

Data loaded!
(31962, 3)


In [ ]:
df.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


In [ ]:
print(df.isnull().sum())
print("-----")
print(df['label'].value_counts())

id       0
label    0
tweet    0
dtype: int64
-----
label
0    29720
1     2242
Name: count, dtype: int64


In [ ]:
print(df['tweet'][0])
print("-----")
print(df['tweet'][1])
print("-----")
print(df['tweet'][2])

 @user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction.   #run
-----
@user @user thanks for #lyft credit i can't use cause they don't offer wheelchair vans in pdx.    #disapointed #getthanked
-----
  bihday your majesty


In [ ]:
import re

def clean_tweet(tweet):
    tweet = re.sub(r'@[\w]*', '', tweet)       # remove @mentions
    tweet = re.sub(r'#[\w]*', '', tweet)        # remove #hashtags
    tweet = re.sub(r'[^a-zA-Z\s]', '', tweet)  # remove numbers & punctuation
    tweet = tweet.lower()                        # make everything lowercase
    tweet = tweet.strip()                        # remove extra spaces
    return tweet

df['clean_tweet'] = df['tweet'].apply(clean_tweet)

print(df['tweet'][0])
print("-----AFTER CLEANING-----")
print(df['clean_tweet'][0])

 @user when a father is dysfunctional and is so selfish he drags his kids into his dysfunction.   #run
-----AFTER CLEANING-----
when a father is dysfunctional and is so selfish he drags his kids into his dysfunction


In [ ]:
df.head()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_tweet'])
y = df['label']

print(X.shape)
print(y.shape)

(31962, 5000)
(31962,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (25569, 5000)
Testing size: (6393, 5000)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("-----")
print(classification_report(y_test, y_pred))

Accuracy: 0.9419677772563742
-----
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      5937
           1       0.90      0.21      0.34       456

    accuracy                           0.94      6393
   macro avg       0.92      0.60      0.65      6393
weighted avg       0.94      0.94      0.92      6393



In [ ]:
model_balanced = LogisticRegression(class_weight='balanced')
model_balanced.fit(X_train, y_train)

y_pred_balanced = model_balanced.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_balanced))
print("-----")
print(classification_report(y_test, y_pred_balanced))

Accuracy: 0.8853433442828094
-----
              precision    recall  f1-score   support

           0       0.98      0.89      0.94      5937
           1       0.36      0.77      0.49       456

    accuracy                           0.89      6393
   macro avg       0.67      0.83      0.71      6393
weighted avg       0.94      0.89      0.90      6393



In [ ]:
def predict_tweet(tweet):
    cleaned = clean_tweet(tweet)
    vectorized = vectorizer.transform([cleaned])
    prediction = model_balanced.predict(vectorized)

    if prediction[0] == 0:
        print("Normal tweet ✅")
    else:
        print("Hate tweet ⚠️")

predict_tweet("I love this beautiful day!")
predict_tweet("I hate you so much")
predict_tweet("This movie was absolutely amazing!")
predict_tweet("you are the worst person ever")

Normal tweet ✅
Hate tweet ⚠️
Normal tweet ✅
Hate tweet ⚠️
